# LEPTOLIC / INT182 รอบ 3 — Repeated nested CV

รันกับ ZIP NHEFS จริงสำเร็จแล้ว ดูผลและข้อจำกัดใน `round3/results/nhefs_20261105/README_TH.md`
Legacy AUC 0.6761 → ใหม่ 0.7054; Brier 0.05653 → 0.05579
แต่ paired Brier 95% CI คร่อมศูนย์ จึงยังไม่ผ่าน primary improvement criterion
Test เคยเปิดเผยแล้ว ไม่ใช่ external validation หรือการยืนยันกับคนไทย

อ่าน `round3/CV_PROTOCOL_TH.md` ก่อนรันซ้ำ ใช้ checkout เต็มและ ZIP ของผู้ใช้
ระดับ 1–3 เป็นช่วงคะแนน development OOF ไม่ใช่เกณฑ์คลินิก

In [1]:
%pip install -r round3/requirements.txt

Note: you may need to restart the kernel to use updated packages.


## ตรวจหลักฐานและ protocol ก่อนฝึก
ไม่ใช้ผล Test เลือกฟีเจอร์/threshold และไม่อ้างว่า Test เดิมเป็นข้อมูลใหม่ที่ไม่เคยเห็น

In [2]:
from pathlib import Path
from IPython.display import display, Markdown
display(Markdown(Path('round3/CV_PROTOCOL_TH.md').read_text(encoding='utf-8')))

# แนวทาง 2: protocol ก่อนรันข้อมูลจริง

สถานะ: โค้ดพัฒนาแล้ว แต่ยังไม่มีผล NHEFS ใหม่ เพราะ runtime ยังบล็อก CDC
รับ ZIP ข้อมูลจาก Colab ได้แล้วโดยใช้ `--data-zip`; ดู [วิธีรัน](README.md)
และ [หลักฐานสถานะล่าสุด](RESULTS_STATUS_TH.md)
เอกสารนี้แทน protocol แบบ single Tune ในการทดลองใหม่ ไม่ได้แก้ผลบันทึกเดิมย้อนหลัง

## คำถามและเกณฑ์ตัดสิน

คงคำถาม reported first diagnosis within 8 years และไม่อ้างใช้กับคนไทยได้
Primary endpoint ของการเปรียบเทียบคือ Test Brier ของโมเดลเดิมลบโมเดลใหม่
ใช้ paired bootstrap 2,000 รอบ สุ่มคนชุดเดียวกันให้ทั้งสองโมเดล
มีหลักฐานภายใน Test ว่า Brier ลดลงเมื่อขอบล่างของช่วง percentile 95% มากกว่า 0
ถ้าช่วงคร่อมศูนย์ ให้สรุปว่ายังพิสูจน์การปรับปรุงตามเกณฑ์นี้ไม่ได้ แม้ point estimate ดีขึ้น
ไม่รับประกันผลบวก และไม่เปลี่ยน metric หลักหลังเห็นผล

Secondary: ROC-AUC, AP, log loss, calibration, recall, precision, specificity และ confusion matrix
รายงานทั้ง threshold 0.5 และกติกาเลือก threshold F2 จาก development OOF สำหรับทั้งโมเดลเดิม/ใหม่
ไม่อ้างว่าการเพิ่ม recall จาก threshold ที่ต่างกันคือการปรับปรุงความสามารถโมเดล
bootstrap เป็น uncertainty ของ fixed fitted models บน Test ไม่ครอบคลุม training/selection uncertainty
Test เดิมเคยเปิดเผยแล้ว ผลจึงเป็น internal evidence ไม่ใช่การพิสูจน์ทั่วไปหรือ external validation

## ข้อมูลและ target

- ใช้ official CDC URLs เดิม เก็บ raw hashes/URLs/bytes ใน data audit
- ID และ split membership สร้างจาก cohort เดิมก่อน additional exclusions แล้วรักษาสมาชิกเดิม
  ไม่เปลี่ยน seed เพื่อย้ายคนที่ผลไม่ดีออกจาก Test
- เพิ่ม conservative guard: label 0 ที่มี recorded death year ก่อนหรือทับปี horizon ถูกคัดออก
  ไม่สมมติวันตายภายในปี กฎนี้ยังต้องยืนยันความหมาย death/status กับ codebook ทางการ
- โมเดลเดิมและใหม่เปรียบเทียบบน guarded cohort เดียวกันเสมอ หากคนเปลี่ยนห้ามเทียบกับ
  AUC 0.676 เดิมโดยอ้างว่าเป็น algorithm improvement โดยตรง
- เก็บเพิ่มเติม exclusion list, join counts และ data-quality summary
- ถ้า baseline features ไม่ครบ หรือไม่มี sleep ใน development จนสร้าง legacy comparator ไม่ได้ ให้หยุด

## Nested repeated validation

1. รวม historical Train/Tune/Calibration เป็น development; historical Test ยังคงแยกไว้
2. Outer repeated stratified 5-fold × 3 repeats ใช้ประเมิน workflow ไม่ใช้เลือกผู้ชนะจาก outer score
3. ภายในแต่ละ outer training ใช้ inner stratified 3-fold เลือก mean Brier ต่ำสุด
4. Candidates 20 แบบ: 4/5 features × (Logistic C=.1/1/10 หรือ Tree depth=2/3, min leaf=50)
   × (ไม่ calibration หรือ sigmoid calibration)
5. Imputer/scaler fit ภายใน training pipeline ทุก fold; sigmoid calibration ใช้ 3-fold
   ภายใน inner training อีกชั้น ด้วย ensemble=False ไม่มีการใช้ inner validation มาฝึก calibrator
6. Feature ที่หายทั้ง training fold ทำให้ candidate/fold นั้นล้มเหลวอย่างเปิดเผย
   ไม่แอบตัด sleep แล้วรายงานว่าเป็นโมเดล 5 features
7. ประเมิน legacy algorithm บน outer validation เดียวกัน โดยแบ่ง outer training เป็น
   Train/Tune/Calibration ในสัดส่วน 55:15:15; ใช้ NumPy solver และ ridge grid เดิม
8. เมื่อครบ outer folds แต่ละคนได้ OOF predictions 3 ค่า เฉลี่ยเป็นคะแนนต่อคน
   เพื่อเลือก F2 threshold และจุดแบ่งระดับ ไม่ถือสาม repeats ว่าเป็นสามคนอิสระ
9. Mean/SD ของ outer folds ใช้แสดงความแกว่ง ไม่ใช้ naive CI ที่ถือ folds เป็นอิสระ
10. เลือก final configuration ด้วย inner CV บน development ทั้งหมด ฝึกแล้วจึงประเมิน Test

ข้อควรอ่าน: legacy workflow กันข้อมูลบางส่วนไว้ tuning/calibration ส่วนใหม่ใช้การ cross-fit
จึงเปรียบเทียบทั้ง workflow ไม่ใช่ผลของอัลกอริทึมเพียงอย่างเดียว
threshold/จุดแบ่งจากค่าเฉลี่ย repeated OOF อาจถ่ายโอนไป final fitted model ได้ไม่พอดี
ต้องรายงานผล Test และการกระจายคะแนนตามจริง ไม่ปรับจุดแบ่งซ้ำจาก Test

## ระดับและการทดลองใช้งาน

ใช้ terciles ของ development OOF scores และห้ามมี development band ว่าง
หากข้อมูลไม่รองรับ ให้คืน level=null พร้อมคำอธิบาย ไม่สร้างสามระดับปลอม
รายงาน n/events/observed rate/Wilson CI ต่อระดับ Test โดยไม่บังคับให้อัตราเรียงกัน
ถ้าระดับใดมีเหตุการณ์น้อย ยังไม่ถือว่าเป็นกลุ่มที่ผ่านการตรวจสอบแล้ว
ระดับเป็นคะแนนสัมพัทธ์ใน historical cohort ไม่ใช่ clinical cutoffs

ส่งออก portable JSON ที่เก็บ sigmoid parameters และ estimator parameters
`round3.predict` รองรับทั้ง legacy round3 JSON และ JSON ของ nested study
ตรวจ export/reload เทียบกับ scikit-learn ทุก 20 candidate types ด้วย software fixtures
fixtures ไม่ใช่ NHEFS และไม่ใช้รายงาน model performance

## คำสั่งและไฟล์ผลจริง

จาก checkout root และ Python environment ที่ติดตั้ง `round3/requirements.txt`:

```bash
python -m unittest discover -s tests -v
python -m round3.cv_study --work-dir round3_work/nested_run_01
python -m round3.cv_study --data-zip /path/to/NHEFS_data_and_layouts.zip --work-dir round3_work/nested_from_zip_01
python -m round3.predict --model round3_work/nested_run_01/model.json --input input.json
```

หากมี ZIP จาก Colab ใช้เส้นทางนี้แทนการดาวน์โหลด CDC:

```bash
python -m round3.cv_study --data-zip /path/NHEFS_data_and_layouts.zip --work-dir round3_work/nested_zip_01
```

ZIP ต้องมีไฟล์ข้อมูลและ layouts ครบ 8 ไฟล์ ยอมรับไฟล์อยู่ในโฟลเดอร์ย่อยได้
ห้ามมีชื่อไฟล์เป้าหมายซ้ำ ระบบตรวจรูปแบบ ASCII records, ID, layout fields และ HTML responses
ก่อนนำเข้าข้อมูล และเก็บ SHA-256/ขนาดไฟล์/จำนวนแถวใน input_archive_manifest.json
การตรวจรูปแบบไม่ใช่การยืนยัน CDC origin โดยอิสระ จึงเก็บ provenance note ไว้ด้วย
ตัวเลขผลจริงยังไม่มีจนกว่าจะได้รับ ZIP หรือเข้าถึง CDC ได้

เก็บ protocol ก่อนดาวน์โหลดและเขียน `study_status.json` ทุกครั้ง
ถ้าล้มเหลวให้เลือก work directory ใหม่เพื่อรักษาหลักฐาน attempt เดิม
เมื่อสำเร็จจึงมี `metrics.json`, `report_TH.md`, `model.json`, `data_audit.json`,
`test_predictions.csv`, `development_oof.csv` โดย raw และ original target outputs อยู่ใน work directory
ไม่เก็บข้อมูลแต่งเป็นผลจริง ไม่รายงานคะแนนใหม่จาก saved outputs เดิม

สิ่งที่ยังต้องทำก่อนอ้างว่างานสมบูรณ์: apply CDC network setting, ตรวจ codebook,
รันข้อมูลจริงครบ, ตรวจจำนวนคนที่ guard คัดออก, อ่านผลและ uncertainty แล้วจึงสรุปว่าดีขึ้นหรือยังไม่ดีขึ้น
หากใช้ ZIP ที่มีข้อมูลครบ ไม่ต้องรอ network setting เพื่อฝึก แต่ยังต้องตรวจ codebook และ provenance


## ฝึกด้วยข้อมูลจริง
เลือก work directory ใหม่สำหรับแต่ละ attempt และกำหนด DATA_ZIP ให้ชี้ ZIP ที่มีข้อมูล/layouts ครบ 8 ไฟล์
เมื่อใช้ ZIP จะไม่ดาวน์โหลด CDC การตรวจรูปแบบไม่ใช่การยืนยันแหล่งกำเนิดโดยอิสระ
ดู input manifest และจำนวนคนที่ถูกคัดออก และยืนยัน codebook ก่อนรับรอง target

In [3]:
import os
from round3.cv_study import run_study
# Set these environment variables or edit the defaults before a new run.
WORK = Path(os.environ.get('LEPTOLIC_WORK_DIR', 'round3_work/nested_zip_01')).resolve()
DATA_ZIP = Path(os.environ.get('LEPTOLIC_DATA_ZIP', 'NHEFS_compressed.zip'))
if not DATA_ZIP.is_file():
    raise FileNotFoundError('Set DATA_ZIP to your eight-file NHEFS ZIP before running')
results = run_study(WORK, data_zip=DATA_ZIP)

## อ่านผลและระดับ
ตรวจจำนวนคน/เหตุการณ์และช่วงความไม่แน่นอนของทุกระดับ ระดับสูงอาจมี observed rate ไม่สูงขึ้นใน Test ขนาดเล็ก ต้องรายงานตามจริง

In [4]:
print(results['test_new'])
print(results['test_legacy_with_same_threshold_policy'])
print(results['paired_test_comparison'])
print(results['test_levels'])
display(Markdown((WORK/'report_TH.md').read_text(encoding='utf-8')))

{'n': 409, 'events': 25, 'roc_auc': 0.7054166666666667, 'brier_score': 0.055794904802818256, 'log_loss': 0.21668041925552595, 'accuracy': 0.38386308068459657, 'precision': 0.08424908424908426, 'recall': 0.92, 'balanced_accuracy': 0.6344791666666667, 'f1': 0.15436241610738255, 'confusion_matrix': {'tn': 134, 'fp': 250, 'fn': 2, 'tp': 23}, 'threshold': 0.04165539699069321, 'observed_rate': 0.061124694376528114, 'mean_predicted': 0.057328047956257254, 'specificity': 0.3489583333333333, 'recall_95ci_wilson': [0.7503389104960468, 0.9777795987151524], 'precision_95ci_wilson': [0.056794077649134855, 0.12324203050103807], 'average_precision': 0.13192156156018403}
{'n': 409, 'events': 25, 'roc_auc': 0.6761458333333333, 'brier_score': 0.056529185516960026, 'log_loss': 0.2227443553335793, 'accuracy': 0.6699266503667481, 'precision': 0.10714285714285714, 'recall': 0.6, 'balanced_accuracy': 0.6372395833333333, 'f1': 0.18181818181818182, 'confusion_matrix': {'tn': 259, 'fp': 125, 'fn': 10, 'tp': 15}

# ผลการทดลอง NHEFS: repeated nested cross-validation

ยังไม่มีหลักฐานเพียงพอว่า Brier ดีขึ้นจริงตามเกณฑ์ที่กำหนด

นี่เป็น internal validation บน Test ที่เคยเปิดเผยแล้ว ไม่ใช่ external validation หรือการยืนยันกับคนไทย

Development 2311 คน / 141 เหตุการณ์; Test 409 คน / 25 เหตุการณ์

| Metric | โมเดลเดิม threshold 0.5 | โมเดลใหม่ threshold OOF |
|---|---:|---:|
| roc_auc | 0.6761458333333333 | 0.7054166666666667 |
| average_precision | 0.12294096861198632 | 0.13192156156018403 |
| brier_score | 0.056529185516960026 | 0.055794904802818256 |
| log_loss | 0.2227443553335793 | 0.21668041925552595 |
| recall | 0.0 | 0.92 |
| precision | None | 0.08424908424908426 |
| specificity | 1.0 | 0.3489583333333333 |

การเปรียบเทียบ recall ข้างบนรวมผลจาก threshold ที่ต่างกัน ต้องอ่าน test_legacy_with_same_threshold_policy และ test_new_at_0_5 ใน metrics.json ประกอบ

ผลต่าง paired bootstrap (ค่าบวกหมายถึงดีขึ้นตามทิศทางที่ระบุ):

```json
{
  "brier_old_minus_new": {
    "ci95": [
      -6.215657078143886e-05,
      0.0016821509447153755
    ],
    "valid_draws": 2000
  },
  "auc_new_minus_old": {
    "ci95": [
      0.0022733569712468675,
      0.06167967812124841
    ],
    "valid_draws": 2000
  },
  "ap_new_minus_old": {
    "ci95": [
      -0.008383520933377034,
      0.0349369312027765
    ],
    "valid_draws": 2000
  },
  "point_differences": {
    "brier_old_minus_new": 0.000734280714141766,
    "auc_new_minus_old": 0.029270833333333357,
    "ap_new_minus_old": 0.00898059294819771
  },
  "primary_supported": false,
  "scope": "conditional on fixed fitted models and historical Test; no training uncertainty or external validation"
}
```

ระดับคะแนน:

```json
[
  {
    "level": 1,
    "n": 155,
    "events": 3,
    "observed_rate": 0.01935483870967742,
    "observed_rate_95ci_wilson": [
      0.0066038767216294765,
      0.0553538696612183
    ],
    "mean_score": 0.03019499157193172
  },
  {
    "level": 2,
    "n": 134,
    "events": 7,
    "observed_rate": 0.05223880597014925,
    "observed_rate_95ci_wilson": [
      0.025532002054082972,
      0.1039026322956776
    ],
    "mean_score": 0.05493090981427678
  },
  {
    "level": 3,
    "n": 120,
    "events": 15,
    "observed_rate": 0.125,
    "observed_rate_95ci_wilson": [
      0.07723497273395374,
      0.1960294020800739
    ],
    "mean_score": 0.09505171671122259
  }
]
```

ค่าเฉลี่ย/SD ของแต่ละ fold ใช้แสดงความแกว่งเท่านั้น folds และ repeats ไม่เป็นอิสระต่อกัน

- Outer legacy roc_auc: mean=0.645982, SD=0.045786
- Outer new roc_auc: mean=0.642611, SD=0.047815
- Outer legacy average_precision: mean=0.122421, SD=0.036445
- Outer new average_precision: mean=0.119015, SD=0.038350
- Outer legacy brier_score: mean=0.056695, SD=0.001012
- Outer new brier_score: mean=0.056492, SD=0.001028

ข้อจำกัด: การเลือก 20 candidates, ข้อมูลย้อนหลัง, reported diagnosis, จำนวนเหตุการณ์น้อย และ conservative death guard ยังต้องยืนยัน codebook

คะแนนทดลองของกลุ่ม NHEFS สหรัฐฯ ในอดีต สำหรับการรายงานวินิจฉัยใน 8 ปี ยังไม่ยืนยันกับคนไทย ไม่ใช่การวินิจฉัยหรือคำแนะนำการรักษา ระดับ 1 ไม่ได้แปลว่าไม่มีโอกาสเกิดเหตุการณ์

## ทดลองกรอกข้อมูล
ตัวอย่างด้านล่างอาจถูกปฏิเสธถ้าอยู่นอก training range; ไม่เหมาะกับผู้เคยได้รับการวินิจฉัยแล้ว
ไม่ใช่เครื่องมือวินิจฉัย/เลือกการรักษา

In [5]:
from round3.predict import predict_file
INPUT = dict(age_years=50, sex_female=0, baseline_sbp=120, baseline_dbp=75, sleep_hours=7)
answer = predict_file(WORK/'model.json', INPUT, previously_diagnosed=False)
print(answer)

{'experimental_probability_8y': 0.06875724048113405, 'level': 2, 'meaning': 'ระดับสัมพัทธ์ตามจุดแบ่งคะแนน development OOF ของ NHEFS', 'warnings': [], 'limitations': 'คะแนนทดลองของกลุ่ม NHEFS สหรัฐฯ ในอดีต สำหรับการรายงานวินิจฉัยใน 8 ปี ยังไม่ยืนยันกับคนไทย ไม่ใช่การวินิจฉัยหรือคำแนะนำการรักษา ระดับ 1 ไม่ได้แปลว่าไม่มีโอกาสเกิดเหตุการณ์', 'features_used': ['age_years', 'sex_female', 'baseline_sbp', 'baseline_dbp'], 'decision_threshold': 0.04165539699069321}


## ไฟล์ผล
เมื่อรันข้อมูลจริงสำเร็จจึงมี report, metrics, model และ predictions ไม่ใช้ผล software fixtures แทนผลจริง

In [6]:
from IPython.display import FileLink
for name in ['report_TH.md', 'metrics.json', 'model.json', 'test_predictions.csv', 'development_oof.csv']:
    display(FileLink(str(WORK/name)))

/workspace/Lebtolic/round3_work/real_nhefs_20261105_notebook/report_TH.md

/workspace/Lebtolic/round3_work/real_nhefs_20261105_notebook/metrics.json

/workspace/Lebtolic/round3_work/real_nhefs_20261105_notebook/model.json

/workspace/Lebtolic/round3_work/real_nhefs_20261105_notebook/test_predictions.csv

/workspace/Lebtolic/round3_work/real_nhefs_20261105_notebook/development_oof.csv